# CALCUALTE GENE-SET BURDEN SCORES 

# GDC SOMATIC MUTATIONS 
This script calculates the rare variant polygenic risk score for gene sets derived from GDC somatic mutations. The gene-level polygenic risk score files are first downloaded from the workspace bucket. Each gene set is then processed in a loop. For each gene in a given set, the script identifies the corresponding gene-level polygenic risk score file and adds its value to a running total. The final sum represents the polygenic risk score for the entire gene set.

In [1]:
# Set bucket environment
# Load packages
import os
import re
import os
import gc
import glob
import pandas as pd 
from functools import reduce

# Set workspace environment variables
os.environ["WORKSPACE_BUCKET"] = "gs://rw-migration-aou-rw-9f879fc4"
os.environ["ARTIFACT_REGISTRY_DOCKER_REPO"] = (
    "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod"
)

# Load the remaining workspace environment settings
%run Setting_Env_Variables.ipynb

Successfully set WORKSPACE_CDR to: wb-silky-artichoke-2408.C2024Q3R9
Found bucket: id=dataproc-staging-aou-rw-9f879fc4, bucketName=dataproc-staging-wb-fast-radish-6415
Found bucket: id=dataproc-temp-aou-rw-9f879fc4, bucketName=dataproc-temp-wb-fast-radish-6415
Found bucket: id=rw-migration-aou-rw-9f879fc4, bucketName=rw-migration-aou-rw-9f879fc4
Found bucket: id=vwb-aou-datasets-controlled-v8r9, bucketName=vwb-aou-datasets-controlled

Variables extracted:
GOOGLE_CLOUD_PROJECT: wb-fast-radish-6415
WORKSPACE_BUCKET: gs://rw-migration-aou-rw-9f879fc4
WORKSPACE_TEMP_BUCKET: NOT FOUND
WORKSPACE_CDR: wb-silky-artichoke-2408.C2024Q3R9

✅ Saved to /home/dataproc/.bashrc
C2024Q3R9 BQ_DATASET
JupyterLab_Spark_cluster_for_AoU_Spark_20261105 DATAPROC_CLUSTER
dataproc-staging-aou-rw-9f879fc4 GCS_BUCKET
dataproc-temp-aou-rw-9f879fc4 GCS_BUCKET
rw-migration-aou-rw-9f879fc4 GCS_BUCKET
vwb-aou-datasets-controlled-v8r9 GCS_BUCKET
GOOGLE_CLOUD_PROJECT = wb-fast-radish-6415
WORKSPACE_BUCKET = gs://rw-migr

In [2]:
import os
import glob
import subprocess

import pandas as pd
import numpy as np
import statsmodels.api as sm


# =============================================================================================================
# Set up directories
# =============================================================================================================

OUTPUT_DIR = "burden_polygenic_score"
SUMMARY_DIR = "logistic_summary"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SUMMARY_DIR, exist_ok=True)

WORKSPACE_BUCKET = os.environ["WORKSPACE_BUCKET"]

GOOGLE_CLOUD_PROJECT = os.environ["GOOGLE_CLOUD_PROJECT"]

# =============================================================================================================
# Copy files from Google Cloud Storage
# =============================================================================================================

# Copy gene-set burden score files
subprocess.run(
    f"gsutil cp {WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_gene_sets/*/*PRS.tsv {OUTPUT_DIR}/",
    shell=True,
    check=True
)

# Copy phenotype file
subprocess.run(
    f"gsutil cp {WORKSPACE_BUCKET}/data/saige_phenotype_files/aml_phenotype_files/EUR_phenotype_file.txt {OUTPUT_DIR}/",
    shell=True,
    check=True
)

# Copy exome burden file
subprocess.run(
    f"gsutil cp {WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_gene_sets/exome/* {OUTPUT_DIR}/",
    shell=True,
    check=True
)

# Copy genomic metrics file
subprocess.run(
    f"cp /home/dataproc/workspace/vwb-aou-datasets-controlled-v8r9/v8/wgs/short_read/snpindel/aux/qc/genomics_metrics_Dec142023_1859_02_tz0000.tsv {OUTPUT_DIR}/",
    shell=True,
    check=True
)


# =============================================================================================================
# List gene-set burden score files
# =============================================================================================================

files = sorted(
    glob.glob(os.path.join(OUTPUT_DIR, "*PRS.tsv"))
)

files = [f for f in files if f != 'burden_polygenic_score/exome-burden-PRS.tsv']
files = [f for f in files if f != 'burden_polygenic_score/Sensitivity-Analysis-PRS.tsv']

print(f"Found {len(files)} gene-set burden score files.")

# =============================================================================================================
# Load files necessary for logistic regression
# =============================================================================================================

# Load phenotype file
pheno = pd.read_csv(
    os.path.join(OUTPUT_DIR, "EUR_phenotype_file.txt"),
    sep="\t"
)

# Load exome burden file
exome = pd.read_csv(
    os.path.join(OUTPUT_DIR, "exome-burden-PRS.tsv"),
    sep="\t"
)

# Load genomic metric file
metrics = pd.read_csv(
    os.path.join(OUTPUT_DIR, "genomics_metrics_Dec142023_1859_02_tz0000.tsv"),
    sep="\t"
)


# Format exome burden file
exome = (
    exome.rename(columns={
        "IID": "person_id",
        "EXOME_SCORE1_SUM": "exome_burden_prs"
    })
    [["person_id", "exome_burden_prs"]]
)

# Format genomic metric file
metrics = (
    metrics.rename(columns={
        "research_id": "person_id"
    })
    [["person_id", "sample_source"]]
)


# =============================================================================================================
# Logistic regression of gene sets for exome burden polygenic risk score
# =============================================================================================================

covariates = [
    "sex", "age",
    "PC1", "PC2", "PC3", "PC4", "PC5",
    "PC6", "PC7", "PC8", "PC9", "PC10"
]

for file in files:

    gene_set = os.path.basename(file).replace("-PRS.tsv", "")

    print(f"\nProcessing file: {gene_set} ...")

    # Load gene-set burden score
    score = pd.read_csv(file, sep="\t")

    # Format gene-set burden score
    score = (
        score.rename(columns={
            "IID": "person_id",
            "TOTAL_SCORE1_SUM": "burden_prs"
        })
        [["person_id", "burden_prs"]]
    )

    # Merge all data frames
    merged_df = (
        pheno
        .merge(score, on="person_id", how="inner")
        .merge(exome, on="person_id", how="inner")
        .merge(metrics, on="person_id", how="inner")
    )

    print(f"Number of individuals: {len(merged_df)}")

    # Skip gene sets with no rare variants
    if merged_df["burden_prs"].max() == 0:

        print(f"Skipping file: {gene_set} (no rare variants)")

        continue

    # Standardize burden scores
    merged_df["burden_prs"] = (
        merged_df["burden_prs"] - merged_df["burden_prs"].mean()
    ) / merged_df["burden_prs"].std()

    merged_df["exome_burden_prs"] = (
        merged_df["exome_burden_prs"] - merged_df["exome_burden_prs"].mean()
    ) / merged_df["exome_burden_prs"].std()

    # Define outcome and predictors
    X = merged_df[
        ["burden_prs", "exome_burden_prs"] + covariates
    ]

    y = merged_df["aml_phenotype"]

    # Add intercept
    X = sm.add_constant(X)

    # Run logistic regression
    logistic = sm.Logit(y, X).fit(disp=False)

    # Extract regression coefficients and statistics
    logistic_summary = pd.DataFrame({
        "Coefficient": logistic.params,
        "Std_Error": logistic.bse,
        "z_value": logistic.tvalues,
        "p_value": logistic.pvalues,
        "OR": np.exp(logistic.params),
        "CI_lower": np.exp(logistic.conf_int()[0]),
        "CI_upper": np.exp(logistic.conf_int()[1])
    })

    # Save summary
    output_file = os.path.join(
        SUMMARY_DIR,
        f"{gene_set}-logistic-summary.tsv"
    )

    logistic_summary.to_csv(
        output_file,
        sep="\t",
        index=True,
        index_label="Variable"
    )

    print(f"Saved results: {output_file}")
    

# Copy exome burden file
subprocess.run(
    f"gsutil cp {OUTPUT_DIR}/* {WORKSPACE_BUCKET}/data/gene_burden_analysis/logistic_results/",
    shell=True,
    check=True
)

print("\nFinished processing all gene sets.")

Copying gs://rw-migration-aou-rw-9f879fc4/data/gene_burden_analysis/burden_gene_sets/exome/exome-burden-PRS.tsv...
Copying gs://rw-migration-aou-rw-9f879fc4/data/gene_burden_analysis/burden_gene_sets/rna_seq_cox_regression/aml-rna-seq-normal-tissue-cox-regression-downregulation-significant-results-PRS.tsv...
Copying gs://rw-migration-aou-rw-9f879fc4/data/gene_burden_analysis/burden_gene_sets/rna_seq_cox_regression/aml-rna-seq-normal-tissue-cox-regression-significant-results-PRS.tsv...
Copying gs://rw-migration-aou-rw-9f879fc4/data/gene_burden_analysis/burden_gene_sets/rna_seq_cox_regression/aml-rna-seq-normal-tissue-cox-regression-upregulation-significant-results-PRS.tsv...
| [4 files][ 30.1 MiB/ 30.1 MiB]                                                
==> NOTE: You are performing a sequence of gsutil operations that may
run significantly faster if you instead use gsutil -m cp ... Please
see the -m section under "gsutil help options" for further information
about when gsutil -m can be

Found 156 gene-set burden score files.

Processing file: B-vs-other-cells-single-cell-downregulation-significant-results ...
Number of individuals: 169971
Saved results: logistic_summary/B-vs-other-cells-single-cell-downregulation-significant-results-logistic-summary.tsv

Processing file: B-vs-other-cells-single-cell-malignant-samples-downregulation-significant-results ...
Number of individuals: 169971
Saved results: logistic_summary/B-vs-other-cells-single-cell-malignant-samples-downregulation-significant-results-logistic-summary.tsv

Processing file: B-vs-other-cells-single-cell-malignant-samples-significant-results ...
Number of individuals: 169971
Saved results: logistic_summary/B-vs-other-cells-single-cell-malignant-samples-significant-results-logistic-summary.tsv

Processing file: B-vs-other-cells-single-cell-malignant-samples-upregulation-significant-results ...
Number of individuals: 169971
Saved results: logistic_summary/B-vs-other-cells-single-cell-malignant-samples-upregulat

/opt/conda/miniconda3/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/opt/conda/miniconda3/lib/python3.11/site-packages/pandas/core/arraylike.py:396: RuntimeWarning: overflow encountered in exp
  result = getattr(ufunc, method)(*inputs, **kwargs)


Saved results: logistic_summary/pDC-vs-other-cells-single-cell-malignant-samples-downregulation-significant-results-logistic-summary.tsv

Processing file: pDC-vs-other-cells-single-cell-malignant-samples-significant-results ...
Number of individuals: 169971
Saved results: logistic_summary/pDC-vs-other-cells-single-cell-malignant-samples-significant-results-logistic-summary.tsv

Processing file: pDC-vs-other-cells-single-cell-malignant-samples-upregulation-significant-results ...
Number of individuals: 169971
Saved results: logistic_summary/pDC-vs-other-cells-single-cell-malignant-samples-upregulation-significant-results-logistic-summary.tsv

Processing file: pDC-vs-other-cells-single-cell-significant-results ...
Number of individuals: 169971
Saved results: logistic_summary/pDC-vs-other-cells-single-cell-significant-results-logistic-summary.tsv

Processing file: pDC-vs-other-cells-single-cell-upregulation-significant-results ...
Number of individuals: 169971
Saved results: logistic_summ

Copying file://burden_polygenic_score/B-vs-other-cells-single-cell-downregulation-significant-results-PRS.tsv [Content-Type=text/tab-separated-values]...
Copying file://burden_polygenic_score/B-vs-other-cells-single-cell-malignant-samples-downregulation-significant-results-PRS.tsv [Content-Type=text/tab-separated-values]...
Copying file://burden_polygenic_score/B-vs-other-cells-single-cell-malignant-samples-significant-results-PRS.tsv [Content-Type=text/tab-separated-values]...
Copying file://burden_polygenic_score/B-vs-other-cells-single-cell-malignant-samples-upregulation-significant-results-PRS.tsv [Content-Type=text/tab-separated-values]...
\ [4 files][ 11.7 MiB/ 11.7 MiB]                                                
==> NOTE: You are performing a sequence of gsutil operations that may
run significantly faster if you instead use gsutil -m cp ... Please
see the -m section under "gsutil help options" for further information
about when gsutil -m can be advantageous.

Copying file:/


Finished processing all gene sets.


In [3]:
!tar -czvf logistic_summary.tar.gz logistic_summary/

logistic_summary/
logistic_summary/B-vs-other-cells-single-cell-downregulation-significant-results-logistic-summary.tsv
logistic_summary/B-vs-other-cells-single-cell-malignant-samples-downregulation-significant-results-logistic-summary.tsv
logistic_summary/B-vs-other-cells-single-cell-malignant-samples-significant-results-logistic-summary.tsv
logistic_summary/B-vs-other-cells-single-cell-malignant-samples-upregulation-significant-results-logistic-summary.tsv
logistic_summary/B-vs-other-cells-single-cell-significant-results-logistic-summary.tsv
logistic_summary/B-vs-other-cells-single-cell-upregulation-significant-results-logistic-summary.tsv
logistic_summary/CTL-vs-other-cells-single-cell-downregulation-significant-results-logistic-summary.tsv
logistic_summary/CTL-vs-other-cells-single-cell-malignant-samples-downregulation-significant-results-logistic-summary.tsv
logistic_summary/CTL-vs-other-cells-single-cell-malignant-samples-significant-results-logistic-summary.tsv
logistic_summary/